### Run Dependencies

In [ ]:
%run Legal_-_Raw_To_Bronze

### Date & Time Format Lists

In [ ]:
# ── Date format lists used for coalesce-based parsing ─────────────────────────
date_formats = [
    "dd-MMM-yy", "dd-MM-yyyy", "yyyy-MM-dd", "d-MMM-yyyy",
    "dd MMM yyyy", "dd/MM/yyyy", "yyyy/MM/dd", "yyyyMMdd",
    "dd MMM yy", "dd MM yyyy", "dd MM yy", "yyyy", "MMM-yy",
]
time_formats = ["HH:mm:ss", "hh:mm:ss a", "hh:mm a", "HH:mm"]

date_time_formats = [
    f"{d} {t}" for d in date_formats for t in time_formats
] + [
    f"{d}'T'{t}" for d in date_formats for t in time_formats
]

print(f"Date formats: {len(date_formats)} | Time formats: {len(time_formats)} | Combined: {len(date_time_formats)}")


### Main Loop — Bronze → Silver (Managed Tables)

In [ ]:
import uuid

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN LOOP  —  BRONZE → SILVER
# ══════════════════════════════════════════════════════════════════════════════
for File_Type in CSV_Files:

    Table_Name = None
    audit_id   = None

    try:
        File_Name    = File_Type.split("/")[-1]
        Table_Name   = re.search(r"([^/]+)(?=\.\w+$)", File_Type).group(1)
        audit_id     = str(uuid.uuid4())
        bronze_table = f"{Bronze_Schema}.{Table_Name}"
        silver_table = f"{Silver_Schema}.{Table_Name}"

        # ── Read from managed Bronze table ────────────────────────────────
        Read_Bronze_df = spark.read.table(bronze_table).withColumn("Comments", lit("-"))

        # ── Retrieve column type lists from pre-built lookup (no Spark jobs)
        Date_Columns      = schema_lookup[(File_Name, "DateType()")]
        DateTime_Columns  = schema_lookup[(File_Name, "TimestampType()")]
        Integer_Columns   = schema_lookup[(File_Name, "IntegerType()")]
        Float_Columns     = schema_lookup[(File_Name, "FloatType()")]
        String_Columns    = schema_lookup[(File_Name, "StringType()")]

        # ── Cast date columns using coalesce over format list ─────────────
        for _dc in Date_Columns:
            Read_Bronze_df = Read_Bronze_df.withColumn(
                _dc,
                coalesce(*[to_date(col(_dc), fmt) for fmt in date_formats]),
            )

        for _dtc in DateTime_Columns:
            Read_Bronze_df = Read_Bronze_df.withColumn(
                _dtc,
                coalesce(*[to_timestamp(col(_dtc), fmt) for fmt in date_time_formats]),
            )

        # ── Cast integer and float columns ────────────────────────────────
        for _ic in Integer_Columns:
            if not _ic.startswith("is_"):
                Read_Bronze_df = Read_Bronze_df.withColumn(_ic, col(_ic).cast(IntegerType()))

        for _fc in Float_Columns:
            Read_Bronze_df = Read_Bronze_df.withColumn(_fc, col(_fc).cast(FloatType()))

        # ── Validation conditions ─────────────────────────────────────────
        main_error_conditions   = []
        normal_error_conditions = []

        for _ic in Integer_Columns:
            if _ic.startswith("is_"):
                normal_error_conditions.append(
                    when(
                        ~col(_ic).cast("string").rlike("^(0|1|True|False|true|false)$"),
                        lit(f"{_ic}: invalid boolean"),
                    )
                )
            else:
                main_error_conditions.append(
                    when(col(_ic).isNull(), lit(f"{_ic}: null"))
                )

        for _fc in Float_Columns:
            main_error_conditions.append(
                when(col(_fc).isNull(), lit(f"{_fc}: null"))
            )

        # ── Apply Comments + Deletion_Flag ────────────────────────────────
        if main_error_conditions:
            Read_Bronze_df = Read_Bronze_df.withColumn(
                "Comments",
                when(
                    concat_ws("; ", *main_error_conditions) == "",
                    lit("-"),
                ).otherwise(concat_ws("; ", *main_error_conditions)),
            )

        Read_Bronze_df = Read_Bronze_df.withColumn(
            "Deletion_Flag",
            when(col("Comments") != "-", lit("Y")).otherwise(lit("N")),
        )

        Error_Log_df   = Read_Bronze_df.filter(col("Deletion_Flag") == "Y")
        Read_Bronze_df = Read_Bronze_df.drop("Deletion_Flag")

        # ── Cache once before counts ───────────────────────────────────────
        Read_Bronze_df.cache()
        Error_Log_df.cache()
        silver_record_count = Read_Bronze_df.count()
        error_record_count  = Error_Log_df.count()

        # ── Write managed Silver table ────────────────────────────────────
        #    mergeSchema only (never overwriteSchema — prevents silent drops)
        Read_Bronze_df.write \
            .mode("overwrite") \
            .format("delta") \
            .option("mergeSchema", "true") \
            .saveAsTable(silver_table)

        Read_Bronze_df.unpersist()
        print(f"[SILVER] Written | table={silver_table} | records={silver_record_count}")

        # ── Write per-table error log ──────────────────────────────────────
        if error_record_count > 0:
            error_table = f"{Error_Schema}.{Table_Name}_error_log"
            Error_Log_df.write \
                .mode("append") \
                .format("delta") \
                .option("mergeSchema", "true") \
                .saveAsTable(error_table)

            json_cols = [col(c) for c in Error_Log_df.columns if c != "Comments"]
            Central_Error_df = Error_Log_df.select(
                expr("uuid()").alias("Error_ID"),
                lit(Table_Name).alias("Source_Table"),
                lit("BRONZE_TO_SILVER").alias("Pipeline_Layer"),
                col("Comments").alias("Error_Message"),
                to_json(struct(*json_cols)).alias("Error_Record_JSON"),
                current_timestamp().alias("Error_Logged_Time"),
            )

            # Central managed error table
            Central_Error_df.write \
                .mode("append") \
                .format("delta") \
                .option("mergeSchema", "true") \
                .saveAsTable(f"{Error_Schema}.{CENTRAL_ERROR_LOG_TABLE}")

            # Warehouse mirror (executemany — fast)
            try:
                _conn   = get_warehouse_conn()
                _cursor = _conn.cursor()
                _params = [
                    (r["Error_ID"], r["Source_Table"], r["Pipeline_Layer"],
                     r["Error_Message"], r["Error_Record_JSON"], r["Error_Logged_Time"])
                    for r in Central_Error_df.collect()
                ]
                _cursor.fast_executemany = True
                _cursor.executemany(
                    f"""INSERT INTO [{WAREHOUSE_SCHEMA}].[{CENTRAL_ERROR_LOG_WH}]
                        (Error_ID,Source_Table,Pipeline_Layer,
                         Error_Message,Error_Record_JSON,Error_Logged_Time)
                        VALUES (?,?,?,?,?,?)""",
                    _params,
                )
                _conn.commit()
                _conn.close()
                print(f"[ERROR LOG] Warehouse updated | table={Table_Name} | rows={len(_params)}")
            except Exception as _we:
                print(f"[ERROR LOG WARN] Warehouse insert failed: {_we}")

        Error_Log_df.unpersist()

        # ── Audit log (SUCCESS) ────────────────────────────────────────────
        try:
            log_audit(
                audit_id          = audit_id,
                source_type       = "Lakehouse",
                destination       = silver_table,
                notebook_name     = "Legal - Bronze_To_Silver",
                layer_name        = "SILVER",
                table_name        = Table_Name,
                records_processed = silver_record_count,
                status            = "SUCCESS",
                error_message     = "",
            )
        except Exception as _ae:
            print(f"[AUDIT WARN] {_ae}")

        # ── Metadata log ──────────────────────────────────────────────────
        update_layer_metadata("SILVER", File_Name, Table_Name, Silver_Schema)

        print(f"[DONE] {Table_Name} | valid={silver_record_count} | errors={error_record_count}")

    except Exception as e:
        try:
            log_audit(
                audit_id          = audit_id or str(uuid.uuid4()),
                source_type       = "Lakehouse",
                destination       = f"{Silver_Schema}.{Table_Name}" if Table_Name else "UNKNOWN",
                notebook_name     = "Legal - Bronze_To_Silver",
                layer_name        = "SILVER",
                table_name        = Table_Name or "UNKNOWN",
                records_processed = 0,
                status            = "FAILED",
                error_message     = str(e),
            )
        except Exception as _ae2:
            print(f"[AUDIT WARN] {_ae2}")
        print(f"[FAILED] {Table_Name} | {e}")
        raise


### Optional Files Backup

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  OPTIONAL FILES BACKUP  —  Bronze → Silver
# ══════════════════════════════════════════════════════════════════════════════
if ENABLE_FILES_BACKUP:
    print("[BACKUP] Starting Silver backup...")

    _silver_tables = [
        (re.search(r"([^/]+)(?=\.\w+$)", f).group(1), Silver_Schema)
        for f in CSV_Files
    ]

    backup_tables_to_files(
        tables      = _silver_tables,
        backup_root = Silver_Backup_Path,
        layer_label = "SILVER_BACKUP",
    )

    _prune_old_backups(Silver_Backup_Path, BACKUP_RETENTION_DAYS)

else:
    print("[BACKUP] Skipped — ENABLE_FILES_BACKUP is False")
